In [2]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns

PARTICIPANT_DATA_PATH = './participant_data/'

In [3]:
from utilities.preprocess import preprocess

index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)

train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))

X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)

In [5]:
y_train

,timeDiff,status
0,75264503.0,0.0
1,75091115.0,0.0
2,75084884.0,0.0
3,75084844.0,0.0
4,75079718.0,0.0
...,...,...
885903,185341.0,1.0
885904,423199.0,1.0
885905,230107.0,1.0
885906,228037.0,1.0


In [3]:
lifelines_train_df.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount,timeDiff,status
0,-0.398558,-2.489321,-2.099994,0.289315,-0.785666,-0.570639,-0.396655,-0.156334,-0.892979,-0.402879,...,-0.079284,-1.733569,-0.398471,-1.254174,-1.577749,-2.370232,1.542022,-0.932965,75264503.0,0.0
1,-0.398558,-2.487828,-2.099970,-0.306477,-0.797257,-0.545239,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.130458,-1.733569,-0.398471,-1.382937,-1.577674,-2.370232,1.542022,-0.932965,75091115.0,0.0
2,-0.398558,-2.487828,-2.097217,-0.306477,-0.797492,-0.276081,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132896,-1.733569,-0.398471,-1.254154,-1.577637,-2.370232,1.542022,-0.932965,75084884.0,0.0
3,-0.398558,-2.487828,-2.097217,-0.306477,-0.797489,-0.274353,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.133089,-1.733569,-0.398471,-0.977241,-1.577637,-2.370232,1.542022,-0.932965,75084844.0,0.0
4,-0.398558,-2.487828,-2.095696,-0.306477,-0.797487,-0.052927,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132848,-1.733569,-0.398471,-1.281682,-1.577600,-2.370232,1.542022,-0.932965,75079718.0,0.0


In [36]:
lifelines_train_df.shape

(885908, 80)

In [43]:
lifelines_train_df["status"].value_counts()
status_0_count = lifelines_train_df["status"].value_counts()[0]
status_1_count = lifelines_train_df["status"].value_counts()[1]

print(f"No of status=0: {status_0_count}")
print(f"No of status=1: {status_1_count}")

No of status=0: 861864
No of status=1: 24044


In [44]:
print(f"% of status=0: {status_0_count/(status_0_count + status_1_count):.2f}")
print(f"% of status=1: {status_1_count/(status_0_count + status_1_count):.2f}")

% of status=0: 0.97
% of status=1: 0.03


In [45]:
##-- First Oversample the minority class --##
import pandas as pd
from sklearn.utils import resample

# Separate majority and minority classes
df_majority = lifelines_train_df[lifelines_train_df['status'] == 0]
df_minority = lifelines_train_df[lifelines_train_df['status'] == 1]

# Over-sample minority class to match majority class size
df_oversampled = resample(df_minority,
                                   replace=True,                 
                                   n_samples=len(df_majority)//5,    
                                   random_state=42)              

# Combine majority class with oversampled minority class
df_balanced = pd.concat([df_majority, df_oversampled])

# Shuffle the dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced dataset shape: {df_balanced.shape}")
laundering_counts = df_balanced['status'].value_counts()
print(laundering_counts)
print(f"\nStatus 0 %: {laundering_counts[0] / (laundering_counts[0] + laundering_counts[1]) * 100:.2f}%")
print(f"Status 1 %: {laundering_counts[1] / (laundering_counts[0] + laundering_counts[1]) * 100:.2f}%")

Balanced dataset shape: (1034236, 80)
status
0.0    861864
1.0    172372
Name: count, dtype: int64

Status 0 %: 83.33%
Status 1 %: 16.67%


In [51]:
df_balanced.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount,timeDiff,status
0,-0.398509,1.695001,0.689370,-1.371254,-0.345753,0.613425,-0.793705,-0.156334,-0.803372,-0.402825,...,0.009835,2.109067,-0.398448,-0.056992,-0.758037,2.008346,-1.525506,-0.615704,58734692.0,0.0
1,-0.107327,0.378040,0.799263,1.354092,-0.693872,0.879344,1.191543,-0.156334,0.182305,0.216558,...,-0.133030,-0.359169,-0.107372,1.039942,0.641736,-0.902107,0.008258,0.137847,9826136.0,0.0
2,-0.398558,-0.833275,-1.143734,-0.591353,-0.556728,-1.026062,-0.793705,-0.156334,-0.713765,-0.402665,...,-0.132632,-0.119609,-0.398471,-1.404857,1.640600,-0.520144,0.008258,-0.589210,52893846.0,0.0
3,0.025070,1.015406,1.063875,-1.073358,-0.624595,0.763318,1.191543,-0.156334,1.078376,0.457364,...,-0.130977,-0.284800,0.024806,1.076725,0.465565,-0.847099,0.008258,1.210734,8964822.0,0.0
4,-0.398558,-0.861169,-1.211335,-1.433531,-0.403069,-0.560185,-0.793705,-0.156334,-0.713765,-0.402825,...,-0.132717,-0.416711,-0.398471,-1.426368,0.923210,-0.604234,0.008258,-0.565744,53401461.0,0.0


In [52]:
len(df_balanced["status"]), len(df_balanced["timeDiff"])

(1034236, 1034236)

In [53]:
model = CoxPHFitter(penalizer=0.1)
model.fit(df_balanced, duration_col='timeDiff', event_col='status')

<lifelines.CoxPHFitter: fitted with 1.03424e+06 total observations, 861864 right-censored observations>

In [54]:
model.print_summary()

<lifelines.CoxPHFitter: fitted with 1.03424e+06 total observations, 861864 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 1.03424e+06
number of events observed = 172372
   partial log-likelihood = -2268960.52
         time fit was run = 2025-09-20 11:44:04 UTC

---
                                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                            
userBorrowSum                       -0.10      0.91      0.00           -0.10           -0.09                0.90                0.91
marketWithdrawAvgAmount             -0.02      0.98      0.00           -0.03           -0.02                0.97                0.98
marketDepositAvgAmountUSD            0.04      1.04      0.00            0.03            0.04                1.03                1.04
sinDayOfMonth                        0.05      1.05      0.00            0.05            0.06                1.05                1.06
userSecondsSinceFirstTransaction     0.02      1.02      0.00            0.02            0.03                1.02                1.03
timeOfDay                            0.01      1.01      0.00            0.01            0.02                1.01                1.02
userActiveDaysWeekly                 0.19      1.21      0.00            0.19            0.19                1.20                1.22
userLiquidationCount                 0.19      1.20      0.00            0.18            0.19                1.20                1.21
userActiveDaysMonthly                0.12      1.13      0.00            0.11            0.12                1.12                1.13
userRepayCount                      -0.11      0.90      0.00           -0.12           -0.10                0.89                0.90
cosQuarter                           0.01      1.01      0.00            0.01            0.02                1.01                1.02
marketRepayAvgAmount                -0.01      0.99      0.00           -0.01           -0.00                0.99                1.00
marketLiquidationSum                 0.01      1.01      0.00            0.00            0.01                1.00                1.01
userLiquidationSum                   0.05      1.06      0.00            0.05            0.06                1.05                1.06
sinDayOfQuarter                      0.03      1.03      0.00            0.03            0.04                1.03                1.04
marketLiquidationCount               0.00      1.00      0.00           -0.00            0.01                1.00                1.01
userRepaySumUSD                     -0.10      0.91      0.00           -0.10           -0.09                0.90                0.91
userWithdrawCount                   -0.03      0.97      0.00           -0.03           -0.02                0.97                0.98
marketRepayCount                     0.03      1.03      0.00            0.03            0.04                1.03                1.04
userLiquidationSumUSD               -0.07      0.93      0.00           -0.07           -0.07                0.93                0.94
marketWithdrawSum                   -0.00      1.00      0.00           -0.01            0.01                0.99                1.01
userWithdrawSumUSD                  -0.04      0.96      0.00           -0.04           -0.03                0.96                0.97
marketBorrowSum                      0.00      1.00      0.00           -0.00            0.01                1.00                1.01
marketLiquidationAvgAmountUSD       -0.02      0.98      0.00           -0.02           -0.01                0.98                0.99
userDepositSumUSD                   -0.04      0.96      0.00   

In [55]:
# from lifelines.utils import concordance_index

# train_risk = model.predict_partial_hazard(X_train)

# # --- c-index using partial hazard as risk scores ---
# c_index = concordance_index(
#     df_balanced["timeDiff"],       # observed times
#     -train_risk,                   # negative because higher risk -> shorter survival
#     df_balanced["status"]          # event indicator
# )
# print(f"C-Index: {c_index:.3f}")

In [ ]:
model.print_summary()

In [5]:
# from lassonet import LassoNetCoxRegressor

# model = LassoNetCoxRegressor(tie_approximation="breslow")

# X_np = X_train.values if hasattr(X_train, "values") else X_train
# y_np = y_train.values if hasattr(y_train, "values") else y_train

# oracle, order, wrong, paths, prob = model.stability_selection(X_np, y_np)

In [7]:
# order

In [16]:
order = [7, 69, 53, 45,  6, 19, 65, 73,  2, 36, 74, 26, 42, 43,  3, 52,  4, 67,
        17,  8, 37, 64, 38, 21, 39,  9,  1, 35, 31, 66, 28, 14, 30, 76, 13, 70,
        40, 33, 24, 23, 15, 49, 50, 55, 27, 34, 16,  0, 12, 61, 20, 18, 48, 29,
        77, 62, 44, 51, 47, 46, 72, 58, 41, 56, 10, 75, 63, 22, 11, 25, 71, 60,
        68, 32, 54, 59,  5, 57]

print(len(order))

78


In [11]:
columns = (lifelines_train_df.keys()).to_list()

In [19]:
# --- select the columns whose index is in order ---
selected_cols = ["timeDiff", "status"]
for i in order:
    selected_cols.append(columns[i])

print(len(selected_cols))

80


In [20]:
new_lifelines_train_df = lifelines_train_df[selected_cols]
new_lifelines_train_df.head()

,timeDiff,status,userLiquidationCount,userBorrowAvgAmountUSD,userBorrowCount,userLiquidationAvgAmount,userActiveDaysWeekly,userLiquidationSumUSD,userDepositCount,logAmountUSD,...,marketRepayAvgAmount,quarter,marketBorrowSumUSD,cosDayOfWeek,sinDayOfWeek,dayOfMonth,dayOfWeek,cosTimeOfDay,timeOfDay,sinTimeOfDay
0,75264503.0,0.0,-0.156334,-0.485221,-0.439298,-0.075548,-0.396655,-0.0872,-0.237205,-1.254174,...,-2.342678,-1.452898,-1.733569,0.978175,1.050891,-0.185892,-1.448052,-0.743173,-0.570639,1.148620
1,75091115.0,0.0,-0.156334,-0.485246,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.382937,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-0.788563,-0.545239,1.108079
2,75084884.0,0.0,-0.156334,-0.485221,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.254154,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.148216,-0.276081,0.575966
3,75084844.0,0.0,-0.156334,-0.485158,-0.439211,-0.075548,-1.190754,-0.0872,-0.239892,-0.977241,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.149720,-0.274353,0.572074
4,75079718.0,0.0,-0.156334,-0.485227,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.281682,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.246935,-0.052927,0.048505


In [21]:
model = CoxPHFitter(penalizer=0.1)
model.fit(new_lifelines_train_df, duration_col='timeDiff', event_col='status')

<lifelines.CoxPHFitter: fitted with 885908 total observations, 861864 right-censored observations>

In [24]:
# model.print_summary()
from lifelines.utils import concordance_index

train_risk = model.predict_partial_hazard(X_train)

# --- c-index using partial hazard as risk scores ---
c_index = concordance_index(
    lifelines_train_df["timeDiff"],       # observed times
    -train_risk,                          # negative because higher risk -> shorter survival
    lifelines_train_df["status"]          # event indicator
)
print(f"C-Index: {c_index:.3f}")

C-Index: 0.814


In [25]:
summary_df = model.summary
summary_df.head()

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
userLiquidationCount,0.178409,1.195314,0.002239,0.174020,0.182798,1.190080,1.200571,0.0,79.676038,0.000000e+00,inf
userBorrowAvgAmountUSD,-0.057935,0.943711,0.003265,-0.064335,-0.051536,0.937691,0.949770,0.0,-17.743118,1.948171e-70,231.572846
userBorrowCount,-0.047917,0.953213,0.003264,-0.054315,-0.041519,0.947134,0.959331,0.0,-14.678561,8.844718e-49,159.629660
userLiquidationAvgAmount,0.033026,1.033577,0.002519,0.028089,0.037963,1.028487,1.038692,0.0,13.111070,2.845553e-39,128.046487
userActiveDaysWeekly,0.062299,1.064281,0.003093,0.056237,0.068361,1.057848,1.070752,0.0,20.142015,3.162034e-90,297.312675
